# 🎯 Solution — Lab 04 : Schema Evolution Automatique avec `dlt` & DuckDB

Ce notebook présente la **solution complète et détaillée** de l'exercice d'application du **Lab 04 (Ingestion avec dlt)**.

---

### 📌 Rappel de la mission :
L'application e-commerce fait évoluer son format de données : les nouvelles commandes contiennent désormais des détails de livraison sous la forme d'un objet imbriqué `delivery` :
```json
"delivery": {
    "carrier": "Chronopost",
    "tracking": "FR8921"
}
```

**Objectifs à atteindre :**
1. Définir une liste `new_orders` avec 2 nouvelles commandes intégrant cet objet imbriqué `delivery`.
2. Exécuter l'ingestion dans la table existante `orders` avec `dev_pipeline.run()`.
3. Observer et valider le **Schema Evolution** automatique opéré par `dlt` (aplatissement en colonnes `delivery__carrier` et `delivery__tracking`, gestion des valeurs nulles pour les anciennes commandes).
4. Interroger les données via SQL et Pandas pour confirmer l'intégrité de la table.

---
## ⚙️ 1. Prérequis : Initialisation du Pipeline et Chargement Initial

Pour que ce notebook de correction soit 100% autonome et exécutable, nous réinitialisons le pipeline `ecommerce_dev` et chargeons le jeu de données initial (les 3 premières commandes sans informations de livraison).

In [ ]:
import dlt
import duckdb
import pandas as pd
from IPython.display import display

# 1. Données initiales (Commandes 1001 à 1003 sans champ delivery)
initial_orders = [
    {
        "order_id": 1001,
        "customer": {"name": "Alice Martin", "city": "Lyon"},
        "amount": 89.90,
        "status": "shipped"
    },
    {
        "order_id": 1002,
        "customer": {"name": "Bob Dupont", "city": "Paris"},
        "amount": 149.00,
        "status": "delivered"
    },
    {
        "order_id": 1003,
        "customer": {"name": "Claire Morel", "city": "Marseille"},
        "amount": 34.50,
        "status": "processing"
    },
]

# 2. Pipeline en dev_mode=True pour isoler l'exécution
dev_pipeline = dlt.pipeline(
    pipeline_name="ecommerce_dev",
    destination="duckdb",
    dataset_name="ecommerce_raw",
    dev_mode=True,
)

# 3. Chargement initial dans la table 'orders'
initial_load_info = dev_pipeline.run(initial_orders, table_name="orders")
print("✅ Chargement initial réussi :")
print(initial_load_info)

Vérifions le schéma initial de la table `orders` : elle ne possède pour l'instant aucune colonne liée à la livraison.

In [ ]:
# Affichage du schéma initial
conn = duckdb.connect(f"{dev_pipeline.pipeline_name}.duckdb")
conn.sql(f"SET search_path = '{dev_pipeline.dataset_name}'")
display(conn.sql("DESCRIBE orders").df()[["column_name", "column_type"]])

---
## 🏋️ 2. Étape 1 : Définir les nouvelles commandes avec l'objet imbriqué `delivery`

Nous construisons la liste `new_orders` avec 2 nouvelles commandes (identifiants `1004` et `1005`).
Chacune possède le sous-dictionnaire `delivery` avec les clés `carrier` et `tracking`.

In [ ]:
# 🎯 Solution Étape 1 : Définition des nouvelles commandes avec l'objet imbriqué
new_orders = [
    {
        "order_id": 1004,
        "customer": {"name": "David Martin", "city": "Nantes"},
        "amount": 75.20,
        "status": "shipped",
        "delivery": {
            "carrier": "Chronopost",
            "tracking": "FR8921"
        }
    },
    {
        "order_id": 1005,
        "customer": {"name": "Emma Bernard", "city": "Bordeaux"},
        "amount": 120.00,
        "status": "processing",
        "delivery": {
            "carrier": "Colissimo",
            "tracking": "FR4530"
        }
    }
]

print(f"✅ {len(new_orders)} nouvelles commandes définies avec l'attribut 'delivery'.")

---
## 🚀 3. Étape 2 : Exécuter le chargement dans la même table `orders`

Nous appelons `dev_pipeline.run()` en ciblant la **même table** (`table_name="orders"`).

C'est ici que la magie de `dlt` opère :
- Au lieu de lever une exception `SchemaMismatchError` ou de rejeter les nouvelles clés (comme le feraient la plupart des ETLs traditionnels), `dlt` compare les données entrantes avec le schéma existant.
- Il détecte deux nouveaux attributs (`delivery.carrier` et `delivery.tracking`).
- Il génère automatiquement les instructions DDL `ALTER TABLE ADD COLUMN` vers la destination (DuckDB ou BigQuery).

In [ ]:
# 🎯 Solution Étape 2 : Exécution de l'ingestion dans la table existante
evolution_load_info = dev_pipeline.run(
    new_orders, 
    table_name="orders"
)

# Affichage du rapport de chargement
print("✅ Ingestion des nouvelles données terminée :")
print(evolution_load_info)

---
## 🔍 4. Étape 3 : Observer le Schema Evolution (Pandas & SQL)

### A. Exploration via `pipeline.dataset().orders.df()`
Affichons l'ensemble des 5 commandes sous forme de DataFrame Pandas.

In [ ]:
# 🎯 Solution Étape 3 : Inspection du DataFrame
dataset = dev_pipeline.dataset()
df_orders = dataset.orders.df()

# Sélection et réordonnancement des colonnes métier pour une lecture claire
business_columns = [
    "order_id",
    "customer__name",
    "customer__city",
    "amount",
    "status",
    "delivery__carrier",
    "delivery__tracking",
]

df_orders[business_columns]

### 💡 Analyse des observations :
1. **Aplatissement automatique (*Flattening*) :**
   - `delivery.carrier` est devenu la colonne `delivery__carrier`.
   - `delivery.tracking` est devenu la colonne `delivery__tracking`.
   - La convention `parent__enfant` est appliquée de manière uniforme.

2. **Rétrocompatibilité & Gestion des valeurs manquantes :**
   - Pour les commandes `1001`, `1002` et `1003` (chargées avant l'évolution), les colonnes de livraison contiennent `NaN` / `None`.
   - Pour les commandes `1004` et `1005`, les valeurs réelles sont correctement peuplées (`Chronopost`, `Colissimo`).
   - Aucune donnée historique n'a été corrompue ou écrasée.

---
### B. Inspection technique des types dans DuckDB
Regardons la structure SQL mise à jour de la table `orders` :

In [ ]:
# Vérification de la nouvelle structure de la table dans DuckDB
display(conn.sql("DESCRIBE orders").df()[["column_name", "column_type", "null"]])

Les colonnes `delivery__carrier` et `delivery__tracking` sont bien typées en `VARCHAR` avec la contrainte `null: YES` (autorisant les valeurs nulles).

---
## 📊 5. Bonus : Exploitation analytique SQL des nouvelles colonnes

Puisque la structure est relationnelle et propre, nous pouvons immédiatement exécuter des requêtes SQL métier combinant clients et transporteurs :

In [ ]:
# Requête SQL analytique : Analyse des commandes par transporteur
query = """
SELECT 
    COALESCE(delivery__carrier, 'Non renseigné (ancien format)') AS transporteur,
    COUNT(*)                                                     AS nb_commandes,
    ROUND(SUM(amount), 2)                                        AS ca_total,
    ROUND(AVG(amount), 2)                                        AS panier_moyen
FROM orders
GROUP BY transporteur
ORDER BY ca_total DESC
"""

display(conn.sql(query).df())

---
## 🎓 Récapitulatif Pédagogique : Pourquoi `dlt` change la donne ?

| Critère | ETL Classique (ex: script custom / vieux connecteurs) | Approche `dlt` |
|:---|:---|:---|
| **Nouveau champ dans le JSON** | 💥 Pipeline crash (`Unrecognized column`) ou donnée silencieusement ignorée | ✅ Détection automatique et ajout de colonne non destructif |
| **Dictionnaires imbriqués** | 🛠️ Nécessite d'écrire du code de parsing JSON ou du SQL `UNNEST` complexe | ✅ Aplatissement automatique (`parent__enfant`) |
| **Maintenance du DDL** | ⏱️ Écriture manuelle de `ALTER TABLE ADD COLUMN ...` | ✅ Inférence de type et exécution DDL transparente |
| **Portabilité** | 🔒 Dépendant du moteur cible | 🌐 Identique sur DuckDB local, BigQuery, Snowflake, PostgreSQL |

---
### 👏 Félicitations !
Vous avez validé avec succès l'exercice d'application sur le **Schema Evolution** de `dlt` !